In [ ]:
FIGURE = 'Rmax_Ts'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
PACKAGE_ROOT = ROOT
import argparse
import json
import os
from pathlib import Path
import re
import shutil
from datetime import datetime
import netCDF4 as nc
import numpy as np
from scipy.special import stdtr
from scipy.stats import rankdata, spearmanr
ROOT = Path(asset_path('asset_bc726fea2e74'))
INPUT = INPUT_DIR
STAGE = PACKAGE_ROOT / 'outputs' / 'runtime' / 'correlations'
OUTPUT = PACKAGE_ROOT / 'outputs' / 'responses'
SEASONS = {'MAM': (3, 4, 5)}
ENGLISH = dict(zip(SEASONS, ('Spring', 'Summer', 'Autumn', 'Winter')))
STARTS = np.arange(2001, 2012, dtype=np.int16)
SCALES = np.arange(1, 25, dtype=np.int16)
BLOCK = 30
FILES = {'CSIF': 'CSIF_0.5deg_2001_2020_monthly_zscore_detrended.nc', 'SPEI': 'SPEI_2001_2020_k01-k24_detrended_monthly.nc', 'SSMIrz': 'SSMIrz_GLEAM_v4.3a_MO_0.5deg_2001_2020_k01-k24_detrended_monthly.nc'}

def log(msg):
    print(f'[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}', flush=True)

def array(v):
    return np.asarray(np.ma.filled(v, np.nan), dtype=np.float64)

def save_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')

def historical_order():
    return {'0_0': [[3, 24], [3, 23], [3, 22], [3, 21], [3, 20], [3, 19], [3, 18], [3, 17], [3, 16], [3, 15], [3, 14], [3, 13], [3, 12], [3, 11], [3, 10], [3, 9], [3, 8], [3, 7], [3, 6], [3, 5], [3, 4], [3, 3], [3, 2], [3, 1], [4, 24], [4, 23], [4, 22], [4, 21], [4, 20], [4, 19], [4, 18], [4, 17], [4, 16], [4, 15], [4, 14], [4, 13], [4, 12], [4, 11], [4, 10], [4, 9], [4, 8], [4, 7], [4, 6], [4, 5], [4, 4], [4, 3], [4, 2], [4, 1], [5, 24], [5, 23], [5, 22], [5, 21], [5, 20], [5, 19], [5, 18], [5, 17], [5, 16], [5, 15], [5, 14], [5, 13], [5, 12], [5, 11], [5, 10], [5, 9], [5, 8], [5, 7], [5, 6], [5, 5], [5, 4], [5, 3], [5, 2], [5, 1]], '0_1': [[6, 24], [6, 23], [6, 22], [6, 21], [6, 20], [6, 19], [6, 18], [6, 17], [6, 16], [6, 15], [6, 14], [6, 13], [6, 12], [6, 11], [6, 10], [6, 9], [6, 8], [6, 7], [6, 6], [6, 5], [6, 4], [6, 3], [6, 2], [6, 1], [7, 24], [7, 23], [7, 22], [7, 21], [7, 20], [7, 19], [7, 18], [7, 17], [7, 16], [7, 15], [7, 14], [7, 13], [7, 12], [7, 11], [7, 10], [7, 9], [7, 8], [7, 7], [7, 6], [7, 5], [7, 4], [7, 3], [7, 2], [7, 1], [8, 24], [8, 23], [8, 22], [8, 21], [8, 20], [8, 19], [8, 18], [8, 17], [8, 16], [8, 15], [8, 14], [8, 13], [8, 12], [8, 11], [8, 10], [8, 9], [8, 8], [8, 7], [8, 6], [8, 5], [8, 4], [8, 3], [8, 2], [8, 1]], '0_2': [[9, 24], [9, 23], [9, 22], [9, 21], [9, 20], [9, 19], [9, 18], [9, 17], [9, 16], [9, 15], [9, 14], [9, 13], [9, 12], [9, 11], [9, 10], [9, 9], [9, 8], [9, 7], [9, 6], [9, 5], [9, 4], [9, 3], [9, 2], [9, 1], [10, 24], [10, 23], [10, 22], [10, 21], [10, 20], [10, 19], [10, 18], [10, 17], [10, 16], [10, 15], [10, 14], [10, 13], [10, 12], [10, 11], [10, 10], [10, 9], [10, 8], [10, 7], [10, 6], [10, 5], [10, 4], [10, 3], [10, 2], [10, 1], [11, 24], [11, 23], [11, 22], [11, 21], [11, 20], [11, 19], [11, 18], [11, 17], [11, 16], [11, 15], [11, 14], [11, 13], [11, 12], [11, 11], [11, 10], [11, 9], [11, 8], [11, 7], [11, 6], [11, 5], [11, 4], [11, 3], [11, 2], [11, 1]], '0_3': [[1, 24], [1, 23], [1, 22], [1, 21], [1, 20], [1, 19], [1, 18], [1, 17], [1, 16], [1, 15], [1, 14], [1, 13], [1, 12], [1, 11], [1, 10], [1, 9], [1, 8], [1, 7], [1, 6], [1, 5], [1, 4], [1, 3], [1, 2], [1, 1], [2, 24], [2, 23], [2, 22], [2, 21], [2, 20], [2, 19], [2, 18], [2, 17], [2, 16], [2, 15], [2, 14], [2, 13], [2, 12], [2, 11], [2, 10], [2, 9], [2, 8], [2, 7], [2, 6], [2, 5], [2, 4], [2, 3], [2, 2], [2, 1], [12, 24], [12, 23], [12, 22], [12, 21], [12, 20], [12, 19], [12, 18], [12, 17], [12, 16], [12, 15], [12, 14], [12, 13], [12, 12], [12, 11], [12, 10], [12, 9], [12, 8], [12, 7], [12, 6], [12, 5], [12, 4], [12, 3], [12, 2], [12, 1]], '1_0': [[3, 24], [3, 23], [3, 22], [3, 21], [3, 20], [3, 19], [3, 18], [3, 17], [3, 16], [3, 15], [3, 14], [3, 13], [3, 12], [3, 11], [3, 10], [3, 9], [3, 8], [3, 7], [3, 6], [3, 5], [3, 4], [3, 3], [3, 2], [3, 1], [4, 24], [4, 23], [4, 22], [4, 21], [4, 20], [4, 19], [4, 18], [4, 17], [4, 16], [4, 15], [4, 14], [4, 13], [4, 12], [4, 11], [4, 10], [4, 9], [4, 8], [4, 7], [4, 6], [4, 5], [4, 4], [4, 3], [4, 2], [4, 1], [5, 24], [5, 23], [5, 22], [5, 21], [5, 20], [5, 19], [5, 18], [5, 17], [5, 16], [5, 15], [5, 14], [5, 13], [5, 12], [5, 11], [5, 10], [5, 9], [5, 8], [5, 7], [5, 6], [5, 5], [5, 4], [5, 3], [5, 2], [5, 1]], '1_1': [[6, 24], [6, 23], [6, 22], [6, 21], [6, 20], [6, 19], [6, 18], [6, 17], [6, 16], [6, 15], [6, 14], [6, 13], [6, 12], [6, 11], [6, 10], [6, 9], [6, 8], [6, 7], [6, 6], [6, 5], [6, 4], [6, 3], [6, 2], [6, 1], [7, 24], [7, 23], [7, 22], [7, 21], [7, 20], [7, 19], [7, 18], [7, 17], [7, 16], [7, 15], [7, 14], [7, 13], [7, 12], [7, 11], [7, 10], [7, 9], [7, 8], [7, 7], [7, 6], [7, 5], [7, 4], [7, 3], [7, 2], [7, 1], [8, 24], [8, 23], [8, 22], [8, 21], [8, 20], [8, 19], [8, 18], [8, 17], [8, 16], [8, 15], [8, 14], [8, 13], [8, 12], [8, 11], [8, 10], [8, 9], [8, 8], [8, 7], [8, 6], [8, 5], [8, 4], [8, 3], [8, 2], [8, 1]], '1_2': [[9, 24], [9, 23], [9, 22], [9, 21], [9, 20], [9, 19], [9, 18], [9, 17], [9, 16], [9, 15], [9, 14], [9, 13], [9, 12], [9, 11], [9, 10], [9, 9], [9, 8], [9, 7], [9, 6], [9, 5], [9, 4], [9, 3], [9, 2], [9, 1], [10, 24], [10, 23], [10, 22], [10, 21], [10, 20], [10, 19], [10, 18], [10, 17], [10, 16], [10, 15], [10, 14], [10, 13], [10, 12], [10, 11], [10, 10], [10, 9], [10, 8], [10, 7], [10, 6], [10, 5], [10, 4], [10, 3], [10, 2], [10, 1], [11, 24], [11, 23], [11, 22], [11, 21], [11, 20], [11, 19], [11, 18], [11, 17], [11, 16], [11, 15], [11, 14], [11, 13], [11, 12], [11, 11], [11, 10], [11, 9], [11, 8], [11, 7], [11, 6], [11, 5], [11, 4], [11, 3], [11, 2], [11, 1]], '1_3': [[12, 1], [1, 2], [2, 2], [12, 2], [1, 3], [2, 3], [12, 3], [1, 4], [2, 4], [12, 4], [1, 5], [2, 5], [12, 5], [1, 6], [2, 6], [12, 6], [1, 7], [2, 7], [12, 7], [1, 8], [2, 8], [12, 8], [1, 9], [2, 9], [12, 9], [1, 10], [2, 10], [12, 10], [1, 11], [2, 11], [12, 11], [1, 12], [2, 12], [12, 12], [1, 13], [2, 13], [12, 13], [1, 14], [2, 14], [12, 14], [1, 15], [2, 15], [12, 15], [1, 16], [2, 16], [12, 16], [1, 17], [2, 17], [12, 17], [1, 18], [2, 18], [12, 18], [1, 19], [2, 19], [12, 19], [1, 20], [2, 20], [12, 20], [1, 21], [2, 21], [12, 21], [1, 22], [2, 22], [12, 22], [1, 23], [2, 23], [12, 23], [1, 24], [2, 24], [12, 24], [1, 1], [2, 1]], '2_0': [[4, 24], [4, 23], [4, 22], [4, 21], [4, 20], [4, 19], [4, 18], [4, 17], [4, 16], [4, 15], [4, 14], [4, 13], [4, 12], [4, 11], [4, 10], [4, 9], [4, 8], [4, 7], [4, 6], [4, 5], [4, 4], [4, 3], [4, 2], [4, 1], [5, 24], [5, 23], [5, 22], [5, 21], [5, 20], [5, 19], [5, 18], [5, 17], [5, 16], [5, 15], [5, 14], [5, 13], [5, 12], [5, 11], [5, 10], [5, 9], [5, 8], [5, 7], [5, 6], [5, 5], [5, 4], [5, 3], [5, 2], [5, 1], [3, 24], [3, 23], [3, 22], [3, 21], [3, 20], [3, 19], [3, 18], [3, 17], [3, 16], [3, 15], [3, 14], [3, 13], [3, 12], [3, 11], [3, 10], [3, 9], [3, 8], [3, 7], [3, 6], [3, 5], [3, 4], [3, 3], [3, 2], [3, 1]], '2_1': [[7, 24], [7, 23], [7, 22], [7, 21], [7, 20], [7, 19], [7, 18], [7, 17], [7, 16], [7, 15], [7, 14], [7, 13], [7, 12], [7, 11], [7, 10], [7, 9], [7, 8], [7, 7], [7, 6], [7, 5], [7, 4], [7, 3], [7, 2], [7, 1], [6, 24], [6, 23], [6, 22], [6, 21], [6, 20], [6, 19], [6, 18], [6, 17], [6, 16], [6, 15], [6, 14], [6, 13], [6, 12], [6, 11], [6, 10], [6, 9], [6, 8], [6, 7], [6, 6], [6, 5], [6, 4], [6, 3], [6, 2], [6, 1], [8, 24], [8, 23], [8, 22], [8, 21], [8, 20], [8, 19], [8, 18], [8, 17], [8, 16], [8, 15], [8, 14], [8, 13], [8, 12], [8, 11], [8, 10], [8, 9], [8, 8], [8, 7], [8, 6], [8, 5], [8, 4], [8, 3], [8, 2], [8, 1]], '2_2': [[9, 24], [9, 23], [9, 22], [9, 21], [9, 20], [9, 19], [9, 18], [9, 17], [9, 16], [9, 15], [9, 14], [9, 13], [9, 12], [9, 11], [9, 10], [9, 9], [9, 8], [9, 7], [9, 6], [9, 5], [9, 4], [9, 3], [9, 2], [9, 1], [10, 24], [10, 23], [10, 22], [10, 21], [10, 20], [10, 19], [10, 18], [10, 17], [10, 16], [10, 15], [10, 14], [10, 13], [10, 12], [10, 11], [10, 10], [10, 9], [10, 8], [10, 7], [10, 6], [10, 5], [10, 4], [10, 3], [10, 2], [10, 1], [11, 24], [11, 23], [11, 22], [11, 21], [11, 20], [11, 19], [11, 18], [11, 17], [11, 16], [11, 15], [11, 14], [11, 13], [11, 12], [11, 11], [11, 10], [11, 9], [11, 8], [11, 7], [11, 6], [11, 5], [11, 4], [11, 3], [11, 2], [11, 1]], '2_3': [[1, 24], [1, 23], [1, 22], [1, 21], [1, 20], [1, 19], [1, 18], [1, 17], [1, 16], [1, 15], [1, 14], [1, 13], [1, 12], [1, 11], [1, 10], [1, 9], [1, 8], [1, 7], [1, 6], [1, 5], [1, 4], [1, 3], [1, 2], [1, 1], [2, 24], [2, 23], [2, 22], [2, 21], [2, 20], [2, 19], [2, 18], [2, 17], [2, 16], [2, 15], [2, 14], [2, 13], [2, 12], [2, 11], [2, 10], [2, 9], [2, 8], [2, 7], [2, 6], [2, 5], [2, 4], [2, 3], [2, 2], [2, 1], [12, 24], [12, 23], [12, 22], [12, 21], [12, 20], [12, 19], [12, 18], [12, 17], [12, 16], [12, 15], [12, 14], [12, 13], [12, 12], [12, 11], [12, 10], [12, 9], [12, 8], [12, 7], [12, 6], [12, 5], [12, 4], [12, 3], [12, 2], [12, 1]], '3_0': [[3, 1], [4, 1], [5, 1], [3, 2], [4, 2], [5, 2], [3, 3], [4, 3], [5, 3], [3, 4], [4, 4], [5, 4], [3, 5], [4, 5], [5, 5], [3, 6], [4, 6], [5, 6], [3, 7], [4, 7], [5, 7], [3, 8], [4, 8], [5, 8], [3, 9], [4, 9], [5, 9], [3, 10], [4, 10], [5, 10], [3, 11], [4, 11], [5, 11], [3, 12], [4, 12], [5, 12], [3, 13], [4, 13], [5, 13], [3, 14], [4, 14], [5, 14], [3, 15], [4, 15], [5, 15], [3, 16], [4, 16], [5, 16], [3, 17], [4, 17], [5, 17], [3, 18], [4, 18], [5, 18], [3, 19], [4, 19], [5, 19], [3, 20], [4, 20], [5, 20], [3, 21], [4, 21], [5, 21], [3, 22], [4, 22], [5, 22], [3, 23], [4, 23], [5, 23], [3, 24], [4, 24], [5, 24]], '3_1': [[6, 1], [7, 1], [8, 1], [6, 2], [7, 2], [8, 2], [6, 3], [7, 3], [8, 3], [6, 4], [7, 4], [8, 4], [6, 5], [7, 5], [8, 5], [6, 6], [7, 6], [8, 6], [6, 7], [7, 7], [8, 7], [6, 8], [7, 8], [8, 8], [6, 9], [7, 9], [8, 9], [6, 10], [7, 10], [8, 10], [6, 11], [7, 11], [8, 11], [6, 12], [7, 12], [8, 12], [6, 13], [7, 13], [8, 13], [6, 14], [7, 14], [8, 14], [6, 15], [7, 15], [8, 15], [6, 16], [7, 16], [8, 16], [6, 17], [7, 17], [8, 17], [6, 18], [7, 18], [8, 18], [6, 19], [7, 19], [8, 19], [6, 20], [7, 20], [8, 20], [6, 21], [7, 21], [8, 21], [6, 22], [7, 22], [8, 22], [6, 23], [7, 23], [8, 23], [6, 24], [7, 24], [8, 24]], '3_2': [[9, 1], [10, 1], [11, 1], [9, 2], [10, 2], [11, 2], [9, 3], [10, 3], [11, 3], [9, 4], [10, 4], [11, 4], [9, 5], [10, 5], [11, 5], [9, 6], [10, 6], [11, 6], [9, 7], [10, 7], [11, 7], [9, 8], [10, 8], [11, 8], [9, 9], [10, 9], [11, 9], [9, 10], [10, 10], [11, 10], [9, 11], [10, 11], [11, 11], [9, 12], [10, 12], [11, 12], [9, 13], [10, 13], [11, 13], [9, 14], [10, 14], [11, 14], [9, 15], [10, 15], [11, 15], [9, 16], [10, 16], [11, 16], [9, 17], [10, 17], [11, 17], [9, 18], [10, 18], [11, 18], [9, 19], [10, 19], [11, 19], [9, 20], [10, 20], [11, 20], [9, 21], [10, 21], [11, 21], [9, 22], [10, 22], [11, 22], [9, 23], [10, 23], [11, 23], [9, 24], [10, 24], [11, 24]], '3_3': [[1, 1], [2, 1], [12, 1], [1, 2], [2, 2], [12, 2], [1, 3], [2, 3], [12, 3], [1, 4], [2, 4], [12, 4], [1, 5], [2, 5], [12, 5], [1, 6], [2, 6], [12, 6], [1, 7], [2, 7], [12, 7], [1, 8], [2, 8], [12, 8], [1, 9], [2, 9], [12, 9], [1, 10], [2, 10], [12, 10], [1, 11], [2, 11], [12, 11], [1, 12], [2, 12], [12, 12], [1, 13], [2, 13], [12, 13], [1, 14], [2, 14], [12, 14], [1, 15], [2, 15], [12, 15], [1, 16], [2, 16], [12, 16], [1, 17], [2, 17], [12, 17], [1, 18], [2, 18], [12, 18], [1, 19], [2, 19], [12, 19], [1, 20], [2, 20], [12, 20], [1, 21], [2, 21], [12, 21], [1, 22], [2, 22], [12, 22], [1, 23], [2, 23], [12, 23], [1, 24], [2, 24], [12, 24]], '4_0': [[3, 1], [4, 1], [5, 1], [3, 2], [4, 2], [5, 2], [3, 3], [4, 3], [5, 3], [3, 4], [4, 4], [5, 4], [3, 5], [4, 5], [5, 5], [3, 6], [4, 6], [5, 6], [3, 7], [4, 7], [5, 7], [3, 8], [4, 8], [5, 8], [3, 9], [4, 9], [5, 9], [3, 10], [4, 10], [5, 10], [3, 11], [4, 11], [5, 11], [3, 12], [4, 12], [5, 12], [3, 13], [4, 13], [5, 13], [3, 14], [4, 14], [5, 14], [3, 15], [4, 15], [5, 15], [3, 16], [4, 16], [5, 16], [3, 17], [4, 17], [5, 17], [3, 18], [4, 18], [5, 18], [3, 19], [4, 19], [5, 19], [3, 20], [4, 20], [5, 20], [3, 21], [4, 21], [5, 21], [3, 22], [4, 22], [5, 22], [3, 23], [4, 23], [5, 23], [3, 24], [4, 24], [5, 24]], '4_1': [[6, 1], [7, 1], [8, 1], [6, 2], [7, 2], [8, 2], [6, 3], [7, 3], [8, 3], [6, 4], [7, 4], [8, 4], [6, 5], [7, 5], [8, 5], [6, 6], [7, 6], [8, 6], [6, 7], [7, 7], [8, 7], [6, 8], [7, 8], [8, 8], [6, 9], [7, 9], [8, 9], [6, 10], [7, 10], [8, 10], [6, 11], [7, 11], [8, 11], [6, 12], [7, 12], [8, 12], [6, 13], [7, 13], [8, 13], [6, 14], [7, 14], [8, 14], [6, 15], [7, 15], [8, 15], [6, 16], [7, 16], [8, 16], [6, 17], [7, 17], [8, 17], [6, 18], [7, 18], [8, 18], [6, 19], [7, 19], [8, 19], [6, 20], [7, 20], [8, 20], [6, 21], [7, 21], [8, 21], [6, 22], [7, 22], [8, 22], [6, 23], [7, 23], [8, 23], [6, 24], [7, 24], [8, 24]], '4_2': [[9, 1], [10, 1], [11, 1], [9, 2], [10, 2], [11, 2], [9, 3], [10, 3], [11, 3], [9, 4], [10, 4], [11, 4], [9, 5], [10, 5], [11, 5], [9, 6], [10, 6], [11, 6], [9, 7], [10, 7], [11, 7], [9, 8], [10, 8], [11, 8], [9, 9], [10, 9], [11, 9], [9, 10], [10, 10], [11, 10], [9, 11], [10, 11], [11, 11], [9, 12], [10, 12], [11, 12], [9, 13], [10, 13], [11, 13], [9, 14], [10, 14], [11, 14], [9, 15], [10, 15], [11, 15], [9, 16], [10, 16], [11, 16], [9, 17], [10, 17], [11, 17], [9, 18], [10, 18], [11, 18], [9, 19], [10, 19], [11, 19], [9, 20], [10, 20], [11, 20], [9, 21], [10, 21], [11, 21], [9, 22], [10, 22], [11, 22], [9, 23], [10, 23], [11, 23], [9, 24], [10, 24], [11, 24]], '4_3': [[1, 1], [2, 1], [12, 1], [1, 2], [2, 2], [12, 2], [1, 3], [2, 3], [12, 3], [1, 4], [2, 4], [12, 4], [1, 5], [2, 5], [12, 5], [1, 6], [2, 6], [12, 6], [1, 7], [2, 7], [12, 7], [1, 8], [2, 8], [12, 8], [1, 9], [2, 9], [12, 9], [1, 10], [2, 10], [12, 10], [1, 11], [2, 11], [12, 11], [1, 12], [2, 12], [12, 12], [1, 13], [2, 13], [12, 13], [1, 14], [2, 14], [12, 14], [1, 15], [2, 15], [12, 15], [1, 16], [2, 16], [12, 16], [1, 17], [2, 17], [12, 17], [1, 18], [2, 18], [12, 18], [1, 19], [2, 19], [12, 19], [1, 20], [2, 20], [12, 20], [1, 21], [2, 21], [12, 21], [1, 22], [2, 22], [12, 22], [1, 23], [2, 23], [12, 23], [1, 24], [2, 24], [12, 24]], '5_0': [[3, 1], [4, 1], [5, 1], [3, 2], [4, 2], [5, 2], [3, 3], [4, 3], [5, 3], [3, 4], [4, 4], [5, 4], [3, 5], [4, 5], [5, 5], [3, 6], [4, 6], [5, 6], [3, 7], [4, 7], [5, 7], [3, 8], [4, 8], [5, 8], [3, 9], [4, 9], [5, 9], [3, 10], [4, 10], [5, 10], [3, 11], [4, 11], [5, 11], [3, 12], [4, 12], [5, 12], [3, 13], [4, 13], [5, 13], [3, 14], [4, 14], [5, 14], [3, 15], [4, 15], [5, 15], [3, 16], [4, 16], [5, 16], [3, 17], [4, 17], [5, 17], [3, 18], [4, 18], [5, 18], [3, 19], [4, 19], [5, 19], [3, 20], [4, 20], [5, 20], [3, 21], [4, 21], [5, 21], [3, 22], [4, 22], [5, 22], [3, 23], [4, 23], [5, 23], [3, 24], [4, 24], [5, 24]], '5_1': [[6, 1], [7, 1], [8, 1], [6, 2], [7, 2], [8, 2], [6, 3], [7, 3], [8, 3], [6, 4], [7, 4], [8, 4], [6, 5], [7, 5], [8, 5], [6, 6], [7, 6], [8, 6], [6, 7], [7, 7], [8, 7], [6, 8], [7, 8], [8, 8], [6, 9], [7, 9], [8, 9], [6, 10], [7, 10], [8, 10], [6, 11], [7, 11], [8, 11], [6, 12], [7, 12], [8, 12], [6, 13], [7, 13], [8, 13], [6, 14], [7, 14], [8, 14], [6, 15], [7, 15], [8, 15], [6, 16], [7, 16], [8, 16], [6, 17], [7, 17], [8, 17], [6, 18], [7, 18], [8, 18], [6, 19], [7, 19], [8, 19], [6, 20], [7, 20], [8, 20], [6, 21], [7, 21], [8, 21], [6, 22], [7, 22], [8, 22], [6, 23], [7, 23], [8, 23], [6, 24], [7, 24], [8, 24]], '5_2': [[9, 1], [10, 1], [11, 1], [9, 2], [10, 2], [11, 2], [9, 3], [10, 3], [11, 3], [9, 4], [10, 4], [11, 4], [9, 5], [10, 5], [11, 5], [9, 6], [10, 6], [11, 6], [9, 7], [10, 7], [11, 7], [9, 8], [10, 8], [11, 8], [9, 9], [10, 9], [11, 9], [9, 10], [10, 10], [11, 10], [9, 11], [10, 11], [11, 11], [9, 12], [10, 12], [11, 12], [9, 13], [10, 13], [11, 13], [9, 14], [10, 14], [11, 14], [9, 15], [10, 15], [11, 15], [9, 16], [10, 16], [11, 16], [9, 17], [10, 17], [11, 17], [9, 18], [10, 18], [11, 18], [9, 19], [10, 19], [11, 19], [9, 20], [10, 20], [11, 20], [9, 21], [10, 21], [11, 21], [9, 22], [10, 22], [11, 22], [9, 23], [10, 23], [11, 23], [9, 24], [10, 24], [11, 24]], '5_3': [[1, 1], [2, 1], [12, 1], [1, 2], [2, 2], [12, 2], [1, 3], [2, 3], [12, 3], [1, 4], [2, 4], [12, 4], [1, 5], [2, 5], [12, 5], [1, 6], [2, 6], [12, 6], [1, 7], [2, 7], [12, 7], [1, 8], [2, 8], [12, 8], [1, 9], [2, 9], [12, 9], [1, 10], [2, 10], [12, 10], [1, 11], [2, 11], [12, 11], [1, 12], [2, 12], [12, 12], [1, 13], [2, 13], [12, 13], [1, 14], [2, 14], [12, 14], [1, 15], [2, 15], [12, 15], [1, 16], [2, 16], [12, 16], [1, 17], [2, 17], [12, 17], [1, 18], [2, 18], [12, 18], [1, 19], [2, 19], [12, 19], [1, 20], [2, 20], [12, 20], [1, 21], [2, 21], [12, 21], [1, 22], [2, 22], [12, 22], [1, 23], [2, 23], [12, 23], [1, 24], [2, 24], [12, 24]], '6_0': [[3, 1], [4, 1], [5, 1], [3, 2], [4, 2], [5, 2], [3, 3], [4, 3], [5, 3], [3, 4], [4, 4], [5, 4], [3, 5], [4, 5], [5, 5], [3, 6], [4, 6], [5, 6], [3, 7], [4, 7], [5, 7], [3, 8], [4, 8], [5, 8], [3, 9], [4, 9], [5, 9], [3, 10], [4, 10], [5, 10], [3, 11], [4, 11], [5, 11], [3, 12], [4, 12], [5, 12], [3, 13], [4, 13], [5, 13], [3, 14], [4, 14], [5, 14], [3, 15], [4, 15], [5, 15], [3, 16], [4, 16], [5, 16], [3, 17], [4, 17], [5, 17], [3, 18], [4, 18], [5, 18], [3, 19], [4, 19], [5, 19], [3, 20], [4, 20], [5, 20], [3, 21], [4, 21], [5, 21], [3, 22], [4, 22], [5, 22], [3, 23], [4, 23], [5, 23], [3, 24], [4, 24], [5, 24]], '6_1': [[6, 1], [7, 1], [8, 1], [6, 2], [7, 2], [8, 2], [6, 3], [7, 3], [8, 3], [6, 4], [7, 4], [8, 4], [6, 5], [7, 5], [8, 5], [6, 6], [7, 6], [8, 6], [6, 7], [7, 7], [8, 7], [6, 8], [7, 8], [8, 8], [6, 9], [7, 9], [8, 9], [6, 10], [7, 10], [8, 10], [6, 11], [7, 11], [8, 11], [6, 12], [7, 12], [8, 12], [6, 13], [7, 13], [8, 13], [6, 14], [7, 14], [8, 14], [6, 15], [7, 15], [8, 15], [6, 16], [7, 16], [8, 16], [6, 17], [7, 17], [8, 17], [6, 18], [7, 18], [8, 18], [6, 19], [7, 19], [8, 19], [6, 20], [7, 20], [8, 20], [6, 21], [7, 21], [8, 21], [6, 22], [7, 22], [8, 22], [6, 23], [7, 23], [8, 23], [6, 24], [7, 24], [8, 24]], '6_2': [[9, 1], [10, 1], [11, 1], [9, 2], [10, 2], [11, 2], [9, 3], [10, 3], [11, 3], [9, 4], [10, 4], [11, 4], [9, 5], [10, 5], [11, 5], [9, 6], [10, 6], [11, 6], [9, 7], [10, 7], [11, 7], [9, 8], [10, 8], [11, 8], [9, 9], [10, 9], [11, 9], [9, 10], [10, 10], [11, 10], [9, 11], [10, 11], [11, 11], [9, 12], [10, 12], [11, 12], [9, 13], [10, 13], [11, 13], [9, 14], [10, 14], [11, 14], [9, 15], [10, 15], [11, 15], [9, 16], [10, 16], [11, 16], [9, 17], [10, 17], [11, 17], [9, 18], [10, 18], [11, 18], [9, 19], [10, 19], [11, 19], [9, 20], [10, 20], [11, 20], [9, 21], [10, 21], [11, 21], [9, 22], [10, 22], [11, 22], [9, 23], [10, 23], [11, 23], [9, 24], [10, 24], [11, 24]], '6_3': [[1, 1], [2, 1], [12, 1], [1, 2], [2, 2], [12, 2], [1, 3], [2, 3], [12, 3], [1, 4], [2, 4], [12, 4], [1, 5], [2, 5], [12, 5], [1, 6], [2, 6], [12, 6], [1, 7], [2, 7], [12, 7], [1, 8], [2, 8], [12, 8], [1, 9], [2, 9], [12, 9], [1, 10], [2, 10], [12, 10], [1, 11], [2, 11], [12, 11], [1, 12], [2, 12], [12, 12], [1, 13], [2, 13], [12, 13], [1, 14], [2, 14], [12, 14], [1, 15], [2, 15], [12, 15], [1, 16], [2, 16], [12, 16], [1, 17], [2, 17], [12, 17], [1, 18], [2, 18], [12, 18], [1, 19], [2, 19], [12, 19], [1, 20], [2, 20], [12, 20], [1, 21], [2, 21], [12, 21], [1, 22], [2, 22], [12, 22], [1, 23], [2, 23], [12, 23], [1, 24], [2, 24], [12, 24]], '7_0': [[3, 1], [4, 1], [5, 1], [3, 2], [4, 2], [5, 2], [3, 3], [4, 3], [5, 3], [3, 4], [4, 4], [5, 4], [3, 5], [4, 5], [5, 5], [3, 6], [4, 6], [5, 6], [3, 7], [4, 7], [5, 7], [3, 8], [4, 8], [5, 8], [3, 9], [4, 9], [5, 9], [3, 10], [4, 10], [5, 10], [3, 11], [4, 11], [5, 11], [3, 12], [4, 12], [5, 12], [3, 13], [4, 13], [5, 13], [3, 14], [4, 14], [5, 14], [3, 15], [4, 15], [5, 15], [3, 16], [4, 16], [5, 16], [3, 17], [4, 17], [5, 17], [3, 18], [4, 18], [5, 18], [3, 19], [4, 19], [5, 19], [3, 20], [4, 20], [5, 20], [3, 21], [4, 21], [5, 21], [3, 22], [4, 22], [5, 22], [3, 23], [4, 23], [5, 23], [3, 24], [4, 24], [5, 24]], '7_1': [[6, 1], [7, 1], [8, 1], [6, 2], [7, 2], [8, 2], [6, 3], [7, 3], [8, 3], [6, 4], [7, 4], [8, 4], [6, 5], [7, 5], [8, 5], [6, 6], [7, 6], [8, 6], [6, 7], [7, 7], [8, 7], [6, 8], [7, 8], [8, 8], [6, 9], [7, 9], [8, 9], [6, 10], [7, 10], [8, 10], [6, 11], [7, 11], [8, 11], [6, 12], [7, 12], [8, 12], [6, 13], [7, 13], [8, 13], [6, 14], [7, 14], [8, 14], [6, 15], [7, 15], [8, 15], [6, 16], [7, 16], [8, 16], [6, 17], [7, 17], [8, 17], [6, 18], [7, 18], [8, 18], [6, 19], [7, 19], [8, 19], [6, 20], [7, 20], [8, 20], [6, 21], [7, 21], [8, 21], [6, 22], [7, 22], [8, 22], [6, 23], [7, 23], [8, 23], [6, 24], [7, 24], [8, 24]], '7_2': [[9, 1], [10, 1], [11, 1], [9, 2], [10, 2], [11, 2], [9, 3], [10, 3], [11, 3], [9, 4], [10, 4], [11, 4], [9, 5], [10, 5], [11, 5], [9, 6], [10, 6], [11, 6], [9, 7], [10, 7], [11, 7], [9, 8], [10, 8], [11, 8], [9, 9], [10, 9], [11, 9], [9, 10], [10, 10], [11, 10], [9, 11], [10, 11], [11, 11], [9, 12], [10, 12], [11, 12], [9, 13], [10, 13], [11, 13], [9, 14], [10, 14], [11, 14], [9, 15], [10, 15], [11, 15], [9, 16], [10, 16], [11, 16], [9, 17], [10, 17], [11, 17], [9, 18], [10, 18], [11, 18], [9, 19], [10, 19], [11, 19], [9, 20], [10, 20], [11, 20], [9, 21], [10, 21], [11, 21], [9, 22], [10, 22], [11, 22], [9, 23], [10, 23], [11, 23], [9, 24], [10, 24], [11, 24]], '7_3': [[1, 1], [2, 1], [12, 1], [1, 2], [2, 2], [12, 2], [1, 3], [2, 3], [12, 3], [1, 4], [2, 4], [12, 4], [1, 5], [2, 5], [12, 5], [1, 6], [2, 6], [12, 6], [1, 7], [2, 7], [12, 7], [1, 8], [2, 8], [12, 8], [1, 9], [2, 9], [12, 9], [1, 10], [2, 10], [12, 10], [1, 11], [2, 11], [12, 11], [1, 12], [2, 12], [12, 12], [1, 13], [2, 13], [12, 13], [1, 14], [2, 14], [12, 14], [1, 15], [2, 15], [12, 15], [1, 16], [2, 16], [12, 16], [1, 17], [2, 17], [12, 17], [1, 18], [2, 18], [12, 18], [1, 19], [2, 19], [12, 19], [1, 20], [2, 20], [12, 20], [1, 21], [2, 21], [12, 21], [1, 22], [2, 22], [12, 22], [1, 23], [2, 23], [12, 23], [1, 24], [2, 24], [12, 24]], '8_0': [[3, 1], [4, 1], [5, 1], [3, 2], [4, 2], [5, 2], [3, 3], [4, 3], [5, 3], [3, 4], [4, 4], [5, 4], [3, 5], [4, 5], [5, 5], [3, 6], [4, 6], [5, 6], [3, 7], [4, 7], [5, 7], [3, 8], [4, 8], [5, 8], [3, 9], [4, 9], [5, 9], [3, 10], [4, 10], [5, 10], [3, 11], [4, 11], [5, 11], [3, 12], [4, 12], [5, 12], [3, 13], [4, 13], [5, 13], [3, 14], [4, 14], [5, 14], [3, 15], [4, 15], [5, 15], [3, 16], [4, 16], [5, 16], [3, 17], [4, 17], [5, 17], [3, 18], [4, 18], [5, 18], [3, 19], [4, 19], [5, 19], [3, 20], [4, 20], [5, 20], [3, 21], [4, 21], [5, 21], [3, 22], [4, 22], [5, 22], [3, 23], [4, 23], [5, 23], [3, 24], [4, 24], [5, 24]], '8_1': [[6, 1], [7, 1], [8, 1], [6, 2], [7, 2], [8, 2], [6, 3], [7, 3], [8, 3], [6, 4], [7, 4], [8, 4], [6, 5], [7, 5], [8, 5], [6, 6], [7, 6], [8, 6], [6, 7], [7, 7], [8, 7], [6, 8], [7, 8], [8, 8], [6, 9], [7, 9], [8, 9], [6, 10], [7, 10], [8, 10], [6, 11], [7, 11], [8, 11], [6, 12], [7, 12], [8, 12], [6, 13], [7, 13], [8, 13], [6, 14], [7, 14], [8, 14], [6, 15], [7, 15], [8, 15], [6, 16], [7, 16], [8, 16], [6, 17], [7, 17], [8, 17], [6, 18], [7, 18], [8, 18], [6, 19], [7, 19], [8, 19], [6, 20], [7, 20], [8, 20], [6, 21], [7, 21], [8, 21], [6, 22], [7, 22], [8, 22], [6, 23], [7, 23], [8, 23], [6, 24], [7, 24], [8, 24]], '8_2': [[9, 1], [10, 1], [11, 1], [9, 2], [10, 2], [11, 2], [9, 3], [10, 3], [11, 3], [9, 4], [10, 4], [11, 4], [9, 5], [10, 5], [11, 5], [9, 6], [10, 6], [11, 6], [9, 7], [10, 7], [11, 7], [9, 8], [10, 8], [11, 8], [9, 9], [10, 9], [11, 9], [9, 10], [10, 10], [11, 10], [9, 11], [10, 11], [11, 11], [9, 12], [10, 12], [11, 12], [9, 13], [10, 13], [11, 13], [9, 14], [10, 14], [11, 14], [9, 15], [10, 15], [11, 15], [9, 16], [10, 16], [11, 16], [9, 17], [10, 17], [11, 17], [9, 18], [10, 18], [11, 18], [9, 19], [10, 19], [11, 19], [9, 20], [10, 20], [11, 20], [9, 21], [10, 21], [11, 21], [9, 22], [10, 22], [11, 22], [9, 23], [10, 23], [11, 23], [9, 24], [10, 24], [11, 24]], '8_3': [[1, 1], [2, 1], [12, 1], [1, 2], [2, 2], [12, 2], [1, 3], [2, 3], [12, 3], [1, 4], [2, 4], [12, 4], [1, 5], [2, 5], [12, 5], [1, 6], [2, 6], [12, 6], [1, 7], [2, 7], [12, 7], [1, 8], [2, 8], [12, 8], [1, 9], [2, 9], [12, 9], [1, 10], [2, 10], [12, 10], [1, 11], [2, 11], [12, 11], [1, 12], [2, 12], [12, 12], [1, 13], [2, 13], [12, 13], [1, 14], [2, 14], [12, 14], [1, 15], [2, 15], [12, 15], [1, 16], [2, 16], [12, 16], [1, 17], [2, 17], [12, 17], [1, 18], [2, 18], [12, 18], [1, 19], [2, 19], [12, 19], [1, 20], [2, 20], [12, 20], [1, 21], [2, 21], [12, 21], [1, 22], [2, 22], [12, 22], [1, 23], [2, 23], [12, 23], [1, 24], [2, 24], [12, 24]], '9_0': [[3, 1], [4, 1], [5, 1], [3, 2], [4, 2], [5, 2], [3, 3], [4, 3], [5, 3], [3, 4], [4, 4], [5, 4], [3, 5], [4, 5], [5, 5], [3, 6], [4, 6], [5, 6], [3, 7], [4, 7], [5, 7], [3, 8], [4, 8], [5, 8], [3, 9], [4, 9], [5, 9], [3, 10], [4, 10], [5, 10], [3, 11], [4, 11], [5, 11], [3, 12], [4, 12], [5, 12], [3, 13], [4, 13], [5, 13], [3, 14], [4, 14], [5, 14], [3, 15], [4, 15], [5, 15], [3, 16], [4, 16], [5, 16], [3, 17], [4, 17], [5, 17], [3, 18], [4, 18], [5, 18], [3, 19], [4, 19], [5, 19], [3, 20], [4, 20], [5, 20], [3, 21], [4, 21], [5, 21], [3, 22], [4, 22], [5, 22], [3, 23], [4, 23], [5, 23], [3, 24], [4, 24], [5, 24]], '9_1': [[6, 1], [7, 1], [8, 1], [6, 2], [7, 2], [8, 2], [6, 3], [7, 3], [8, 3], [6, 4], [7, 4], [8, 4], [6, 5], [7, 5], [8, 5], [6, 6], [7, 6], [8, 6], [6, 7], [7, 7], [8, 7], [6, 8], [7, 8], [8, 8], [6, 9], [7, 9], [8, 9], [6, 10], [7, 10], [8, 10], [6, 11], [7, 11], [8, 11], [6, 12], [7, 12], [8, 12], [6, 13], [7, 13], [8, 13], [6, 14], [7, 14], [8, 14], [6, 15], [7, 15], [8, 15], [6, 16], [7, 16], [8, 16], [6, 17], [7, 17], [8, 17], [6, 18], [7, 18], [8, 18], [6, 19], [7, 19], [8, 19], [6, 20], [7, 20], [8, 20], [6, 21], [7, 21], [8, 21], [6, 22], [7, 22], [8, 22], [6, 23], [7, 23], [8, 23], [6, 24], [7, 24], [8, 24]], '9_2': [[9, 1], [10, 1], [11, 1], [9, 2], [10, 2], [11, 2], [9, 3], [10, 3], [11, 3], [9, 4], [10, 4], [11, 4], [9, 5], [10, 5], [11, 5], [9, 6], [10, 6], [11, 6], [9, 7], [10, 7], [11, 7], [9, 8], [10, 8], [11, 8], [9, 9], [10, 9], [11, 9], [9, 10], [10, 10], [11, 10], [9, 11], [10, 11], [11, 11], [9, 12], [10, 12], [11, 12], [9, 13], [10, 13], [11, 13], [9, 14], [10, 14], [11, 14], [9, 15], [10, 15], [11, 15], [9, 16], [10, 16], [11, 16], [9, 17], [10, 17], [11, 17], [9, 18], [10, 18], [11, 18], [9, 19], [10, 19], [11, 19], [9, 20], [10, 20], [11, 20], [9, 21], [10, 21], [11, 21], [9, 22], [10, 22], [11, 22], [9, 23], [10, 23], [11, 23], [9, 24], [10, 24], [11, 24]], '9_3': [[1, 1], [2, 1], [12, 1], [1, 2], [2, 2], [12, 2], [1, 3], [2, 3], [12, 3], [1, 4], [2, 4], [12, 4], [1, 5], [2, 5], [12, 5], [1, 6], [2, 6], [12, 6], [1, 7], [2, 7], [12, 7], [1, 8], [2, 8], [12, 8], [1, 9], [2, 9], [12, 9], [1, 10], [2, 10], [12, 10], [1, 11], [2, 11], [12, 11], [1, 12], [2, 12], [12, 12], [1, 13], [2, 13], [12, 13], [1, 14], [2, 14], [12, 14], [1, 15], [2, 15], [12, 15], [1, 16], [2, 16], [12, 16], [1, 17], [2, 17], [12, 17], [1, 18], [2, 18], [12, 18], [1, 19], [2, 19], [12, 19], [1, 20], [2, 20], [12, 20], [1, 21], [2, 21], [12, 21], [1, 22], [2, 22], [12, 22], [1, 23], [2, 23], [12, 23], [1, 24], [2, 24], [12, 24]], '10_0': [[3, 1], [4, 1], [5, 1], [3, 2], [4, 2], [5, 2], [3, 3], [4, 3], [5, 3], [3, 4], [4, 4], [5, 4], [3, 5], [4, 5], [5, 5], [3, 6], [4, 6], [5, 6], [3, 7], [4, 7], [5, 7], [3, 8], [4, 8], [5, 8], [3, 9], [4, 9], [5, 9], [3, 10], [4, 10], [5, 10], [3, 11], [4, 11], [5, 11], [3, 12], [4, 12], [5, 12], [3, 13], [4, 13], [5, 13], [3, 14], [4, 14], [5, 14], [3, 15], [4, 15], [5, 15], [3, 16], [4, 16], [5, 16], [3, 17], [4, 17], [5, 17], [3, 18], [4, 18], [5, 18], [3, 19], [4, 19], [5, 19], [3, 20], [4, 20], [5, 20], [3, 21], [4, 21], [5, 21], [3, 22], [4, 22], [5, 22], [3, 23], [4, 23], [5, 23], [3, 24], [4, 24], [5, 24]], '10_1': [[6, 1], [7, 1], [8, 1], [6, 2], [7, 2], [8, 2], [6, 3], [7, 3], [8, 3], [6, 4], [7, 4], [8, 4], [6, 5], [7, 5], [8, 5], [6, 6], [7, 6], [8, 6], [6, 7], [7, 7], [8, 7], [6, 8], [7, 8], [8, 8], [6, 9], [7, 9], [8, 9], [6, 10], [7, 10], [8, 10], [6, 11], [7, 11], [8, 11], [6, 12], [7, 12], [8, 12], [6, 13], [7, 13], [8, 13], [6, 14], [7, 14], [8, 14], [6, 15], [7, 15], [8, 15], [6, 16], [7, 16], [8, 16], [6, 17], [7, 17], [8, 17], [6, 18], [7, 18], [8, 18], [6, 19], [7, 19], [8, 19], [6, 20], [7, 20], [8, 20], [6, 21], [7, 21], [8, 21], [6, 22], [7, 22], [8, 22], [6, 23], [7, 23], [8, 23], [6, 24], [7, 24], [8, 24]], '10_2': [[9, 1], [10, 1], [11, 1], [9, 2], [10, 2], [11, 2], [9, 3], [10, 3], [11, 3], [9, 4], [10, 4], [11, 4], [9, 5], [10, 5], [11, 5], [9, 6], [10, 6], [11, 6], [9, 7], [10, 7], [11, 7], [9, 8], [10, 8], [11, 8], [9, 9], [10, 9], [11, 9], [9, 10], [10, 10], [11, 10], [9, 11], [10, 11], [11, 11], [9, 12], [10, 12], [11, 12], [9, 13], [10, 13], [11, 13], [9, 14], [10, 14], [11, 14], [9, 15], [10, 15], [11, 15], [9, 16], [10, 16], [11, 16], [9, 17], [10, 17], [11, 17], [9, 18], [10, 18], [11, 18], [9, 19], [10, 19], [11, 19], [9, 20], [10, 20], [11, 20], [9, 21], [10, 21], [11, 21], [9, 22], [10, 22], [11, 22], [9, 23], [10, 23], [11, 23], [9, 24], [10, 24], [11, 24]], '10_3': [[1, 1], [2, 1], [12, 1], [1, 2], [2, 2], [12, 2], [1, 3], [2, 3], [12, 3], [1, 4], [2, 4], [12, 4], [1, 5], [2, 5], [12, 5], [1, 6], [2, 6], [12, 6], [1, 7], [2, 7], [12, 7], [1, 8], [2, 8], [12, 8], [1, 9], [2, 9], [12, 9], [1, 10], [2, 10], [12, 10], [1, 11], [2, 11], [12, 11], [1, 12], [2, 12], [12, 12], [1, 13], [2, 13], [12, 13], [1, 14], [2, 14], [12, 14], [1, 15], [2, 15], [12, 15], [1, 16], [2, 16], [12, 16], [1, 17], [2, 17], [12, 17], [1, 18], [2, 18], [12, 18], [1, 19], [2, 19], [12, 19], [1, 20], [2, 20], [12, 20], [1, 21], [2, 21], [12, 21], [1, 22], [2, 22], [12, 22], [1, 23], [2, 23], [12, 23], [1, 24], [2, 24], [12, 24]]}

def centered_ranks(data):
    valid = np.isfinite(data).all(axis=0)
    ranks = rankdata(np.where(valid[None], data, 0.0), axis=0, method='average')
    centered = (ranks - ranks.mean(axis=0)).astype(np.float32)
    ss = np.sum(centered * centered, axis=0, dtype=np.float64)
    return (centered, ss, valid)

def correlate(x, yrank, yss, yvalid, yfinite):
    xr, xss, valid = centered_ranks(x)
    valid &= yvalid & (xss > 0) & (yss > 0)
    cross = np.sum(xr * yrank, axis=0, dtype=np.float64)
    with np.errstate(divide='ignore', invalid='ignore'):
        r = cross / 9.0 / np.sqrt(xss / 9.0) / np.sqrt(yss / 9.0)
    r = np.where(valid, np.clip(r, -1.0, 1.0), np.nan)
    with np.errstate(divide='ignore', invalid='ignore'):
        t = r * np.sqrt(8.0 / ((1.0 + r) * (1.0 - r)))
    p = 2 * stdtr(8, -np.abs(t))
    n = (np.isfinite(x) & yfinite).sum(axis=0).astype(np.uint8)
    return (r, p, n)

def create_file(path, pathway, lat, lon, monthly):
    ds = open_nc(str(path), 'w', format='NETCDF4')
    ds.createDimension('window', 11)
    ds.createDimension('lat', len(lat))
    ds.createDimension('lon', len(lon))
    for name, values, dtype, dim in [('window', np.arange(11), 'i2', 'window'), ('window_start_year', STARTS, 'i2', 'window'), ('window_end_year', STARTS + 9, 'i2', 'window'), ('lat', lat, 'f8', 'lat'), ('lon', lon, 'f8', 'lon')]:
        v = ds.createVariable(name, dtype, (dim,))
        v[:] = values
    ds.analysis_period = '2001-2020'
    ds.input_sif = str(INPUT / FILES['CSIF'])
    ds.input_moisture = str(INPUT / FILES[pathway])
    ds.detrending = 'Existing full-period 2001-2020 OLS residuals, separately by calendar month, grid cell, and moisture timescale. No within-window refitting.'
    ds.correlation_method = 'Spearman correlation calculated independently for each calendar month using ten cross-year same-month pairs per ten-year window.'
    ds.missing_data_rule = 'All ten paired observations must be finite; constant series yield NaN. Matches the original luccpy is_skipna=False default.'
    ds.window_definition = '2001-2010, 2002-2011, ..., 2011-2020.'
    ds.timescales = '1-24 months'
    ds.created_at = datetime.now().isoformat(timespec='seconds')
    ds.completed_lat_rows = 0
    if monthly:
        ds.createDimension('month', 12)
        ds.createDimension('scale', 24)
        ds.createVariable('month', 'i2', ('month',))[:] = np.arange(1, 13)
        ds.createVariable('scale', 'i2', ('scale',))[:] = SCALES
        dims = ('window', 'month', 'scale', 'lat', 'lon')
        chunks = (1, 1, 1, min(BLOCK, len(lat)), min(120, len(lon)))
        specs = [('r', 'f8', np.nan), ('pvalue', 'f8', np.nan), ('valid_n', 'u1', 255)]
    else:
        ds.createDimension('season', len(SEASONS))
        ds.createDimension('name_strlen', 3)
        ds.createVariable('season', 'i2', ('season',))[:] = np.arange(len(SEASONS))
        sv = ds.createVariable('season_name', 'S1', ('season', 'name_strlen'))
        sv[:] = nc.stringtochar(np.array(list(SEASONS), dtype='S3'))
        dims = ('window', 'season', 'lat', 'lon')
        chunks = (1, 1, min(BLOCK, len(lat)), min(120, len(lon)))
        specs = [('max_r', 'f8', np.nan), ('max_abs_r', 'f8', np.nan), ('opt_scale', 'i2', -9999), ('opt_month', 'i2', -9999), ('pvalue_at_opt', 'f8', np.nan), ('valid_n_at_opt', 'u1', 255)]
        ds.scale_selection = 'Select the largest absolute r among 3 calendar months x 24 timescales = 72 candidates per season; retain the signed r, selected month and timescale.'
        ds.tie_rule = 'Strict greater-than absolute comparison in original SPEI seasonal-folder os.listdir order; same saved traversal applied to both pathways.'
        ds.candidates_per_season = 72
        ds.monthly_candidate_count = 288
    for name, dtype, fill in specs:
        ds.createVariable(name, dtype, dims, fill_value=fill, zlib=True, complevel=2, shuffle=True, chunksizes=chunks)
    return ds

def process(pathway, order, resume=False, limit_blocks=None):
    monthly_path = STAGE / f'SIF_{pathway}_monthly_correlations_k01-k24.nc'
    seasonal_path = STAGE / f'SIF_{pathway}_season_maxcorr_k01-k24.nc'
    if monthly_path.exists() != seasonal_path.exists():
        raise RuntimeError('Incomplete output pair; investigate before rerunning.')
    with open_nc(str(INPUT / FILES['CSIF'])) as sy, open_nc(str(INPUT / FILES[pathway])) as sx:
        lat, lon = (array(sy['lat'][:]), array(sy['lon'][:]))
        assert np.allclose(lat, array(sx['lat'][:]), rtol=0, atol=1e-08)
        assert np.allclose(lon, array(sx['lon'][:]), rtol=0, atol=1e-08)
        assert list(np.asarray(sx['scale'][:])) == list(SCALES)
        assert sy['CSIF_zscore'].shape[0] == sx[pathway].shape[1] == 240
        if monthly_path.exists():
            if not resume:
                raise FileExistsError('Output exists; use --resume to continue without overwriting.')
            dm, ds = (open_nc(str(monthly_path), 'r+'), open_nc(str(seasonal_path), 'r+'))
            start_lat = min(int(dm.completed_lat_rows), int(ds.completed_lat_rows))
        else:
            dm = create_file(monthly_path, pathway, lat, lon, True)
            ds = create_file(seasonal_path, pathway, lat, lon, False)
            start_lat = 0
        priority = np.full((11, len(SEASONS), 12, 24), 999, dtype=np.int16)
        for w in range(11):
            for si in range(len(SEASONS)):
                for rank, (month, k) in enumerate(order[f'{w}_{si}']):
                    priority[w, si, month - 1, k - 1] = rank
        processed = 0
        for a in range(start_lat, len(lat), BLOCK):
            b = min(a + BLOCK, len(lat))
            log(f'{pathway}: latitude rows {a}:{b}, preparing same-month ranks')
            y = array(sy['CSIF_zscore'][:, a:b, :])
            yranks, ystats = ({}, {})
            for w in range(11):
                for m in range(12):
                    ids = m + 12 * np.arange(w, w + 10)
                    ranks, ss, good = centered_ranks(y[ids])
                    yranks[w, m], ystats[w, m] = (ranks, (ss, good, np.isfinite(y[ids])))
            shape = (11, len(SEASONS), b - a, len(lon))
            best = np.full(shape, np.nan)
            bk, bm = (np.full(shape, -9999, np.int16), np.full(shape, -9999, np.int16))
            bp, bn = (np.full(shape, np.nan), np.zeros(shape, np.uint8))
            brank = np.full(shape, 999, np.int16)
            for ki, k in enumerate(SCALES):
                x = array(sx[pathway][ki, :, a:b, :])
                rr = np.full((11, 12, b - a, len(lon)), np.nan)
                pp = np.full_like(rr, np.nan)
                nn = np.zeros(rr.shape, np.uint8)
                for w in range(11):
                    for m in range(12):
                        ids = m + 12 * np.arange(w, w + 10)
                        ss, good, yfinite = ystats[w, m]
                        r, p, n = correlate(x[ids], yranks[w, m], ss, good, yfinite)
                        rr[w, m], pp[w, m], nn[w, m] = (r, p, n)
                        si = next((i for i, months in enumerate(SEASONS.values()) if m + 1 in months), None)
                        if si is None:
                            continue
                        pri = priority[w, si, m, ki]
                        win = np.isfinite(r) & (~np.isfinite(best[w, si]) | (np.abs(r) > np.abs(best[w, si])) | (np.abs(r) == np.abs(best[w, si])) & (pri < brank[w, si]))
                        best[w, si] = np.where(win, r, best[w, si])
                        bk[w, si] = np.where(win, k, bk[w, si])
                        bm[w, si] = np.where(win, m + 1, bm[w, si])
                        bp[w, si] = np.where(win, p, bp[w, si])
                        bn[w, si] = np.where(win, n, bn[w, si])
                        brank[w, si] = np.where(win, pri, brank[w, si])
                dm['r'][:, :, ki, a:b, :] = rr
                dm['pvalue'][:, :, ki, a:b, :] = pp
                dm['valid_n'][:, :, ki, a:b, :] = nn
                if k % 6 == 0:
                    log(f'{pathway}: rows {a}:{b}, scale {k}/24 complete')
            for name, values in [('max_r', best), ('max_abs_r', np.abs(best)), ('opt_scale', bk), ('opt_month', bm), ('pvalue_at_opt', bp), ('valid_n_at_opt', bn)]:
                ds[name][:, :, a:b, :] = values
            dm.completed_lat_rows = ds.completed_lat_rows = b
            dm.sync()
            ds.sync()
            processed += 1
            log(f'{pathway}: rows {b}/{len(lat)} saved')
            if limit_blocks and processed >= limit_blocks:
                break
        complete = int(dm.completed_lat_rows) == len(lat)
        dm.close()
        ds.close()
    return complete

def validate(pathway, order):
    monthly_path = STAGE / f'SIF_{pathway}_monthly_correlations_k01-k24.nc'
    seasonal_path = STAGE / f'SIF_{pathway}_season_maxcorr_k01-k24.nc'
    rng = np.random.default_rng(20261021)
    monthly_checks, seasonal_checks, tie_checks = (0, 0, 0)
    maximum_error = 0.0
    changed = total = 0
    with open_nc(str(monthly_path)) as dm, open_nc(str(seasonal_path)) as ds, open_nc(str(INPUT / FILES['CSIF'])) as sy, open_nc(str(INPUT / FILES[pathway])) as sx:
        assert dm.completed_lat_rows == ds.completed_lat_rows == len(dm.dimensions['lat'])
        for a in range(0, 360, BLOCK):
            b = min(a + BLOCK, 360)
            r = array(ds['max_r'][:, :, a:b, :])
            k = np.ma.filled(ds['opt_scale'][:, :, a:b, :], -9999)
            m = np.ma.filled(ds['opt_month'][:, :, a:b, :], -9999)
            assert np.all(~np.isfinite(r) | (k >= 1) & (k <= 24))
            assert np.allclose(array(ds['max_abs_r'][:, :, a:b, :]), np.abs(r), equal_nan=True)
            for si, months in enumerate(SEASONS.values()):
                assert np.all(~np.isfinite(r[:, si]) | np.isin(m[:, si], months))
        base_map = array(ds['max_r'][0, 0, :, :])
        pixels = np.argwhere(np.isfinite(base_map))
        if len(pixels) == 0:
            raise AssertionError('No valid pixels')
        chosen = pixels[rng.choice(len(pixels), min(18, len(pixels)), replace=False)]
        for a, b in chosen:
            y = array(sy['CSIF_zscore'][:, a, b])
            xx = array(sx[pathway][:, :, a, b])
            for w, si in [(0, 0), (5, 0), (10, 0), (3, 0)]:
                best = np.nan
                bestk = bestm = -9999
                for month, k in order[f'{w}_{si}']:
                    ids = month - 1 + 12 * np.arange(w, w + 10)
                    x1, y1 = (xx[k - 1, ids], y[ids])
                    if np.isfinite(x1).all() and np.isfinite(y1).all() and (np.ptp(x1) > 0) and (np.ptp(y1) > 0):
                        ref = spearmanr(x1 - x1.mean(), y1 - y1.mean())
                        actual = float(dm['r'][w, month - 1, k - 1, a, b])
                        error = abs(actual - ref.statistic)
                        maximum_error = max(maximum_error, error)
                        assert error < 2e-14, (pathway, w, month, k, error)
                        assert abs(float(dm['pvalue'][w, month - 1, k - 1, a, b]) - ref.pvalue) < 2e-13
                        monthly_checks += 1
                        if np.isnan(best) or abs(ref.statistic) > abs(best):
                            best, bestk, bestm = (ref.statistic, k, month)
                        elif abs(ref.statistic) == abs(best):
                            tie_checks += 1
                actual = float(ds['max_r'][w, si, a, b])
                assert np.isclose(actual, best, rtol=0, atol=2e-14, equal_nan=True)
                assert int(ds['opt_scale'][w, si, a, b]) == bestk, (pathway, w, si, 'scale', int(ds['opt_scale'][w, si, a, b]), bestk)
                assert int(ds['opt_month'][w, si, a, b]) == bestm, (pathway, w, si, 'month', int(ds['opt_month'][w, si, a, b]), bestm)
                seasonal_checks += 1
    result = {'pathway': pathway, 'passed': True, 'monthly_scalar_scipy_comparisons': monthly_checks, 'seasonal_literal_original_loop_comparisons': seasonal_checks, 'observed_equal_abs_candidates': tie_checks, 'maximum_absolute_r_error': maximum_error, 'all_seasons_month_membership_valid': True, 'n_per_monthly_correlation': 10}
    save_json(STAGE / f'QA_{pathway}.json', result)
    return result

def publish(pathway):
    OUTPUT.mkdir(parents=True, exist_ok=True)
    for name in [f'SIF_{pathway}_monthly_correlations_k01-k24.nc', f'SIF_{pathway}_season_maxcorr_k01-k24.nc', f'QA_{pathway}.json']:
        dest = OUTPUT / name
        if dest.exists():
            raise FileExistsError(f'Publication already exists: {dest}')
        shutil.copy2(STAGE / name, dest)
    folder = OUTPUT / 'original_input_adapter' / pathway
    folder.mkdir(parents=True, exist_ok=True)
    with open_nc(str(STAGE / f'SIF_{pathway}_season_maxcorr_k01-k24.nc')) as src:
        for w, start in enumerate(STARTS):
            for si, season in enumerate(SEASONS):
                dest = folder / f'max_corr_{start}-{start + 9}_{ENGLISH[season]}.nc'
                if dest.exists():
                    raise FileExistsError(dest)
                ds = open_nc('adapter.nc', 'w', memory=15000000)
                ds.createDimension('lat', 360)
                ds.createDimension('lon', 720)
                ds.createVariable('lat', 'f8', ('lat',))[:] = src['lat'][:]
                ds.createVariable('lon', 'f8', ('lon',))[:] = src['lon'][:]
                for out, source, dtype, fill in [('max_correlation', 'max_r', 'f8', np.nan), ('time_scale', 'opt_scale', 'i2', -9999), ('month', 'opt_month', 'i2', -9999)]:
                    ds.createVariable(out, dtype, ('lat', 'lon'), fill_value=fill)[:] = src[source][w, si]
                ds.method = src.scale_selection
                ds.source_file = str(OUTPUT / f'SIF_{pathway}_season_maxcorr_k01-k24.nc')
                buffer = ds.close()
                dest.write_bytes(buffer)

def self_test():
    rng = np.random.default_rng(51)
    for ties in [False, True]:
        x = rng.normal(size=(10, 3, 17))
        y = rng.normal(size=(10, 3, 17))
        if ties:
            x = np.round(x)
            y = np.round(y)
        x[0, 0, 0] = np.nan
        y[2, 0, 1] = np.nan
        x[:, 0, 2] = 1
        y[:, 0, 3] = 2
        yr, ss, valid = centered_ranks(y)
        r, p, n = correlate(x, yr, ss, valid, np.isfinite(y))
        for a in range(3):
            for b in range(17):
                xx, yy = (x[:, a, b], y[:, a, b])
                count = (np.isfinite(xx) & np.isfinite(yy)).sum()
                assert n[a, b] == count
                if count < 10 or np.ptp(xx) == 0 or np.ptp(yy) == 0:
                    assert np.isnan(r[a, b])
                else:
                    ref = spearmanr(xx, yy)
                    assert abs(r[a, b] - ref.statistic) < 2e-14
                    assert abs(p[a, b] - ref.pvalue) < 2e-13
    log('Self-test passed: independent scipy correlations, ties, constant series, incomplete pairs.')

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--pathway', choices=['SPEI', 'SSMIrz', 'both'], default='both')
    parser.add_argument('--resume', action='store_true', default=True)
    parser.add_argument('--limit-blocks', type=int)
    parser.add_argument('--validate-only', action='store_true')
    parser.add_argument('--publish', action='store_true')
    parser.add_argument('--self-test', action='store_true')
    args = parser.parse_args()
    if args.self_test:
        self_test()
        return
    STAGE.mkdir(parents=True, exist_ok=True)
    order_file = STAGE / 'original_candidate_traversal.json'
    if order_file.exists():
        order = json.loads(order_file.read_text(encoding='utf-8'))
    else:
        order = historical_order()
        save_json(order_file, order)
    pathways = ['SPEI', 'SSMIrz'] if args.pathway == 'both' else [args.pathway]
    all_qa = []
    for pathway in pathways:
        complete = args.validate_only or process(pathway, order, args.resume, args.limit_blocks)
        if complete:
            qa = validate(pathway, order)
            all_qa.append(qa)
            log(f'Independent QA passed: {qa}')
            if args.publish:
                publish(pathway)
        else:
            log('Partial computation saved; continue with --resume.')
    all_qa = [json.loads((STAGE / f'QA_{p}.json').read_text(encoding='utf-8')) for p in ['SPEI', 'SSMIrz'] if (STAGE / f'QA_{p}.json').exists()]
    save_json(STAGE / 'run_summary.json', {'created_at': datetime.now().isoformat(), 'selected_preprocessing': 'Keep existing 2001-2020 calendar-month OLS detrending.', 'selection': 'Monthly correlations followed by seasonal maximum across 72 month-timescale candidates.', 'outputs': str(OUTPUT), 'qa': all_qa})
if __name__ == '__main__':
    main()